In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

import pandas as pd

df = dataset["train"].to_pandas()


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [ ]:
print(dataset)
print(dataset['train'].features)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
{'flags': Value('string'), 'instruction': Value('string'), 'category': Value('string'), 'intent': Value('string'), 'response': Value('string')}


In [ ]:
print(dataset['train'][0])

{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [ ]:
import pandas as pd

df = dataset["train"].to_pandas()

df["instruction_words"] = df["instruction"].str.split().str.len()
df["response_words"] = df["response"].str.split().str.len()

print("Instruction:")
print(df["instruction_words"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nResponse:")
print(df["response_words"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

Instruction:
count    26872.000000
mean         8.690979
std          2.605004
min          1.000000
50%          9.000000
90%         12.000000
95%         13.000000
99%         14.000000
max         16.000000
Name: instruction_words, dtype: float64

Response:
count    26872.000000
mean       104.789037
std         52.966204
min          9.000000
50%         90.000000
90%        173.000000
95%        206.000000
99%        300.000000
max        402.000000
Name: response_words, dtype: float64


In [ ]:
intent_stats = (
    df.groupby("intent")["response_words"]
      .agg(["count", "mean", "median", "max"])
      .sort_values("median", ascending=False)
)

intent_stats

,count,mean,median,max
intent,,,,
check_refund_policy,997,263.610832,276.0,402
cancel_order,998,178.511022,187.0,325
recover_password,995,160.993970,162.0,293
switch_account,1000,142.476000,138.0,302
set_up_shipping_address,997,135.457372,133.0,292
delivery_options,995,130.191960,131.0,274
check_payment_methods,999,127.757758,130.0,252
get_refund,997,105.154463,97.0,371
create_account,997,99.230692,95.0,266


In [ ]:
for intent in [
    "cancel_order",
    "check_refund_policy",
    "recover_password",
    "contact_human_agent"
]:
    print(f"\n{'='*80}")
    print(f"INTENT: {intent}")
    print(f"{'='*80}")

    samples = (
        df[df["intent"] == intent]
        .sample(5, random_state=42)
    )

    for i, row in samples.iterrows():
        print("\nCustomer:", row["instruction"])
        print("Response:", row["response"])


INTENT: cancel_order

Customer: I cannot afford purchase {{Order Number}}
Response: I understand the financial constraints you are facing and the need to cancel purchase {{Order Number}}. Our goal is to assist you in the best way possible. To proceed with the cancellation, please follow these steps:

1. Sign in to Your Account: Access our {{Online Company Portal Info}} by signing in with your credentials.

2. Navigate to Your Orders: Once you are signed in, locate the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.

3. Find the Relevant Purchase: Look for the purchase associated with the order number {{Order Number}} and click on it for more details.

4. Initiate Cancellation: You will see an option labeled '{{Online Order Interaction}}.' Please select this option to begin the cancellation process.

5. Complete the Cancellation: Follow any further instructions or prompts to complete the cancellation successfully.

If you encounter any issues or have any quest

In [ ]:
# Exact duplicate instructions
instruction_counts = df["instruction"].value_counts()

print("Unique instructions:", instruction_counts.nunique())
print("Total unique instruction values:", df["instruction"].nunique())
print("Duplicated instruction rows:", (instruction_counts > 1).sum())
print("Max repetitions of one instruction:", instruction_counts.max())

Unique instructions: 8
Total unique instruction values: 24635
Duplicated instruction rows: 989
Max repetitions of one instruction: 8


In [ ]:
# How many different responses exist for each instruction?
response_variation = (
    df.groupby("instruction")["response"]
      .nunique()
      .sort_values(ascending=False)
)

print(response_variation.head(20))

instruction
can i order from {{Delivery City}}                          8
is it possible to place an order from {{Delivery City}}?    8
can I order from {{Delivery City}}?                         8
d uship to {{Delivery City}}                                8
can I place an order from {{Delivery City}}?                8
delieries to {{Delivery City}}                              8
is it possible to order from {{Delivery City}}?             8
is it possible to order from {{Delivery City}} ?            8
is it possible to order from {{Delivery City}}              8
deliveries to {{Delivery City}}                             8
can i place an order from {{Delivery City}}                 8
do you shp to {{Delivery City}}?                            8
do uship to {{Delivery City}}                               8
do you ship to {{Delivery City}}?                           8
do ya deliver to {{Delivery City}}                          8
is it possible to place an order from {{Delivery City}}   

In [ ]:
# Instructions that have multiple different responses
multi_response = response_variation[response_variation > 1]

print("Instructions with multiple different responses:",
      len(multi_response))

print("\nDistribution:")
print(multi_response.value_counts().sort_index())

Instructions with multiple different responses: 989

Distribution:
response
2    396
3    293
4     46
5    203
6     25
7      2
8     24
Name: count, dtype: int64


In [ ]:
instruction_intent_variation = (
    df.groupby("instruction")["intent"]
      .nunique()
      .sort_values(ascending=False)
)

print("Instructions with multiple intents:",
      (instruction_intent_variation > 1).sum())

print("\nMax number of intents for one instruction:",
      instruction_intent_variation.max())

Instructions with multiple intents: 0

Max number of intents for one instruction: 1


In [ ]:
conflicting_instructions = (
    instruction_intent_variation[
        instruction_intent_variation > 1
    ]
)

print(conflicting_instructions.head(20))

Series([], Name: intent, dtype: int64)


improved baseline

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
)

import pandas as pd

df = dataset["train"].to_pandas()


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

In [ ]:
df_model = df.copy()
df_model = df_model[
    ["instruction", "category", "intent", "response"]
].copy()

df_model = df_model.rename(
    columns={
        "instruction": "customer_message"
    }
)

df_model.head()


,customer_message,category,intent,response
0,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [ ]:
df_model = df.copy()
df_model = df_model[
    ["instruction", "category", "intent", "response"]
].copy()

df_model = df_model.rename(
    columns={
        "instruction": "customer_message"
    }
)

from sklearn.model_selection import train_test_split

unique_instructions = df_model["customer_message"].unique()

train_instructions, temp_instructions = train_test_split(
    unique_instructions,
    test_size=0.20,
    random_state=42
)

val_instructions, test_instructions = train_test_split(
    temp_instructions,
    test_size=0.50,
    random_state=42
)

train_df = df_model[
    df_model["customer_message"].isin(train_instructions)
].copy()

val_df = df_model[
    df_model["customer_message"].isin(val_instructions)
].copy()

test_df = df_model[
    df_model["customer_message"].isin(test_instructions)
].copy()

from datasets import Dataset

train_intent = train_df[["customer_message", "response"]].copy()
val_intent = val_df[["customer_message", "response"]].copy()
test_intent = test_df[["customer_message", "response"]].copy()



In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_intent,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_intent,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_intent,
    preserve_index=False
)

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

In [ ]:


model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def Tokenize_response(row):
  prompt = f"Customer: {row['customer_message']}\n Response:"
  target = row["response"] + tokenizer.eos_token

  prompt_tokens = tokenizer(
        prompt,
        add_special_tokens=False
    )

  full_tokens = tokenizer(
        prompt + target,
        truncation=True,
        max_length=512,
        add_special_tokens=False
    )

  input_ids = full_tokens["input_ids"]
  attention_mask = full_tokens["attention_mask"]

  prompt_length = len(prompt_tokens["input_ids"])

  labels = [-100] * prompt_length + input_ids[prompt_length:]

  return {
      "input_ids": input_ids,
      "attention_mask": attention_mask,
      "labels": labels
  }



In [ ]:
train_dataset = train_dataset.map(
    Tokenize_response,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    Tokenize_response,
    remove_columns=val_dataset.column_names
)

test_dataset = test_dataset.map(
    Tokenize_response,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/21482 [00:00<?, ? examples/s]

Map:   0%|          | 0/2663 [00:00<?, ? examples/s]

Map:   0%|          | 0/2727 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float32
)

model.config.pad_token_id = tokenizer.pad_token_id

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen_response_05b",

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=16,

    learning_rate=1e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,

    fp16=False,

    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

In [ ]:
trainer.train(
    resume_from_checkpoint="/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_05b/checkpoint-1343"

)

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Epoch,Training Loss,Validation Loss
2,0.533737,0.604009
3,0.450735,0.607608


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=4029, training_loss=0.33149189403436413, metrics={'train_runtime': 9628.8551, 'train_samples_per_second': 6.693, 'train_steps_per_second': 0.418, 'total_flos': 1.9495707980576256e+16, 'train_loss': 0.33149189403436413, 'epoch': 3.0})

In [ ]:
import os
print(os.listdir("./qwen_response_05b"))

['checkpoint-1343']


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

shutil.copytree(
    "./qwen_response_05b/checkpoint-1343",
    "/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_05b/checkpoint-1343",
    dirs_exist_ok=True
)

In [ ]:
import os

print(os.listdir(
    "/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_05b/checkpoint-1343"
))

In [4]:
!ls "/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent"

01_EDA.ipynb			    qwen_intent_05b_final.zip
02_ModelBenchmark.ipynb		    qwen_response_05b
03_QLoRA_response_generation.ipynb  qwen_response_05b_final.zip
04_model_improvement.ipynb


In [6]:
import zipfile

zip_path = "/content/drive/MyDrive/Start_LLM/AI_Customer_Support_Automation_Agent/qwen_response_05b_final.zip"
extract_path = "/content/qwen_response_05b"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [7]:
!find /content/qwen_response_05b -maxdepth 2 -type f | head -30

/content/qwen_response_05b/qwen_response_05b_final/generation_config.json
/content/qwen_response_05b/qwen_response_05b_final/model.safetensors
/content/qwen_response_05b/qwen_response_05b_final/config.json
/content/qwen_response_05b/qwen_response_05b_final/chat_template.jinja
/content/qwen_response_05b/qwen_response_05b_final/tokenizer.json
/content/qwen_response_05b/qwen_response_05b_final/tokenizer_config.json


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/qwen_response_05b/qwen_response_05b_final"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32
).to("cuda")

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896, padding_idx=151643)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
   

In [11]:
def generate_response(customer_message):
    prompt = f"""You are a bank customer support assistant.

Reply to the customer in 1-2 short sentences.
Use natural, grammatically correct English.
Answer only what the customer asked.
Do not use greetings, apologies, numbered lists, generic customer-service phrases, or unnecessary explanations.

Customer: i need help cancelling purchase {{Order Number}}
Response:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

In [12]:
customer_message = "i need help cancelling purchase {{Order Number}}"

print(generate_response(customer_message))

We're here for you to us regarding your request to cancel purchase {{Order Number}}. I understand that you may have encountered difficulties during the cancellation process and would like assistance with this matter. Rest assured, I'm here to guide you through the steps and ensure a smooth resolution. To proceed with the cancellation, please follow these instructions:

1. Sign into Your Account: Access our platform by logging into your {{Online Company Portal Info}}.
2. Navigate to Your Orders: Once logged in, locate the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Find Your Purchase: Look for the purchase associated with the order number {{Order Number}} and click on it.
4. Initiate Cancellation: Within the details of the purchase, there should be an option labeled '{{Online Order Interaction}}'. Please select this option.
5. Confirm Cancellation: The system might ask for confirmation or additional information related to the cancellation. Kindly provide

In [15]:
def generate_response(customer_message):
    prompt = f"""You are a bank customer support assistant.

Reply to the customer in 1-2 short sentences.
Use natural, grammatically correct English.
Answer only what the customer asked.
Do not use greetings, apologies, numbered lists, generic customer-service phrases, or unnecessary explanations.

Customer: i need help cancelling purchase {{Order Number}}
Response:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

In [16]:
customer_message = "i need help cancelling purchase {{Order Number}}"

print(generate_response(customer_message))

We're here for you to us for assistance with canceling your purchase with the order number {{Order Number}}. I understand that you may have encountered difficulties or confusion during the cancellation process, and I'm here to guide you through it. To ensure a smooth resolution, please follow these steps:

1. Sign into Your Account: Access our platform by logging into your {{Online Company Portal Info}}.
2. Locate Your Orders: Once logged in, navigate to either the '{{Online Order Interaction}}' or '{{Online Order Interaction}}' section.
3. Identify the Relevant Purchase: Look for the order number {{Order Number}} within this section and select it.
4. Initiate Cancellation: You should find an option labeled '{{Cancel Purchase}}' associated with your order. Please click on it.
5. Complete Any Further Steps: The system might prompt you for additional information or feedback. Kindly provide any necessary details as requested.

If you encounter any challenges along the way or have further 

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float32
).to("cuda")

base_model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [18]:
customer_message = "i need help cancelling purchase {{Order Number}}"

prompt = f"""You are a customer support assistant.

Write a clear, concise, and natural response to the customer.
Answer the customer's request directly.
Avoid unnecessary greetings, apologies, repetition, and filler.
Do not mention these instructions.

Customer: {customer_message}
Response:"""

inputs = base_tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        eos_token_id=base_tokenizer.eos_token_id,
        pad_token_id=base_tokenizer.pad_token_id
    )

generated = base_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(generated.strip())

I'm sorry, but I can't assist with that. Is there anything else you'd like assistance with? 

Note: This response is designed to be helpful while maintaining a professional tone appropriate for customer service interactions. It avoids any potential misunderstandings or miscommunications. If you have any other questions or concerns, please feel free to ask. Let me know if you would like me to provide more information on this topic. 

I am here to assist you with your inquiries and resolve any issues you may encounter. Please let me know how I can assist you better. 

Please note that I will keep your personal information confidential and only use it as directed by our terms of service. Thank you for choosing us as your source of knowledge and assistance. Have a great day! 
I am here to assist you with your inquiries and resolve any issues you may encounter. Please let me know how I can assist you better. 
Please note that I will keep your personal information confidential and only use i

In [19]:
def generate_response(customer_message):
    prompt = f"""You are a bank customer support assistant.

Reply to the customer in 1-2 short sentences.
Use natural, grammatically correct English.
Answer only what the customer asked.
Do not use greetings, apologies, numbered lists, generic customer-service phrases, or unnecessary explanations.

Customer: i need help cancelling purchase {{Order Number}}
Response:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return generated.strip()

In [20]:
customer_message = "i need help cancelling purchase {{Order Number}}"

print(generate_response(customer_message))

We're here for you to us regarding your request to cancel purchase {{Order Number}}. I understand that you may have encountered difficulties during the cancellation process and would like assistance with this matter. Rest assured, I'm here to guide you through the steps and ensure a smooth resolution. To proceed with the cancellation, please follow these instructions:

1. Sign into Your Account: Access our platform by logging into
